In [3]:
import requests

url = "https://api.bybit.com/v5/market/orderbook?category=spot&symbol=BTCUSDT&limit=100"
response = requests.get(url, timeout=10)
print(" Connected!" if response.status_code == 200 else f"Failed: {response.status_code}")

 Connected!


 **Orderbook Analysis (Bybit)**

In [7]:
import requests
import pandas as pd
import numpy as np

print("=== Crypto Orderbook Analysis ===")
print("Attempting to fetch live data...\n")

# Try Bybit first (often less restricted than Binance)
try:
    url = "https://api.bybit.com/v5/market/orderbook?category=spot&symbol=BTCUSDT&limit=100"
    response = requests.get(url, timeout=10)
    data = response.json()
    
    if data['retCode'] == 0:
        # Bybit data structure
        bids_data = data['result']['b']
        asks_data = data['result']['a']
        
        bids = pd.DataFrame(bids_data, columns=['Price', 'Quantity'], dtype=float)
        asks = pd.DataFrame(asks_data, columns=['Price', 'Quantity'], dtype=float)
        
        best_bid = bids['Price'].max()
        best_ask = asks['Price'].min()
        spread = best_ask - best_bid
        mid_price = (best_ask + best_bid) / 2
        spread_bps = (spread / mid_price) * 10000
        
        # Liquidity depth within 0.1%
        price_tolerance = 0.001
        bid_depth = bids[bids['Price'] >= best_bid * (1 - price_tolerance)]
        ask_depth = asks[asks['Price'] <= best_ask * (1 + price_tolerance)]
        
        bid_liquidity = (bid_depth['Price'] * bid_depth['Quantity']).sum()
        ask_liquidity = (ask_depth['Price'] * ask_depth['Quantity']).sum()
        
        print(" LIVE DATA from Bybit")
        print(f"Symbol: BTC/USDT")
        print(f"Best Bid: ${best_bid:,.2f}")
        print(f"Best Ask: ${best_ask:,.2f}")
        print(f"Spread: ${spread:.2f} ({spread_bps:.2f} basis points)")
        print(f"Mid Price: ${mid_price:,.2f}")
        print(f"Liquidity Depth (top 0.1%): ${bid_liquidity:,.0f} (Bid) | ${ask_liquidity:,.0f} (Ask)")
        
        live_data_used = True
        
    else:
        raise Exception("Bybit API returned error")
        
except Exception as e:
    print(f"Could not fetch live data: {e}")
    print("\n Using simulated data for demonstration...\n")
    
    # Simulated orderbook data (teaches same concepts)
    np.random.seed(42)
    
    # Create realistic BTC orderbook
    base_price = 85000
    bid_prices = [base_price - x * 10 for x in range(100)]
    ask_prices = [base_price + x * 10 for x in range(100)]
    
    bid_quantities = np.random.exponential(0.5, 100)
    ask_quantities = np.random.exponential(0.5, 100)
    
    bids = pd.DataFrame({'Price': bid_prices, 'Quantity': bid_quantities})
    asks = pd.DataFrame({'Price': ask_prices, 'Quantity': ask_quantities})
    
    best_bid = bids['Price'].max()
    best_ask = asks['Price'].min()
    spread = best_ask - best_bid
    mid_price = (best_ask + best_bid) / 2
    spread_bps = (spread / mid_price) * 10000
    
    # Liquidity depth within 0.1%
    price_tolerance = 0.001
    bid_depth = bids[bids['Price'] >= best_bid * (1 - price_tolerance)]
    ask_depth = asks[asks['Price'] <= best_ask * (1 + price_tolerance)]
    
    bid_liquidity = (bid_depth['Price'] * bid_depth['Quantity']).sum()
    ask_liquidity = (ask_depth['Price'] * ask_depth['Quantity']).sum()
    
    print(" SIMULATED DATA (API blocked - using realistic model)")
    print(f"Symbol: BTC/USDT (simulated)")
    print(f"Best Bid: ${best_bid:,.2f}")
    print(f"Best Ask: ${best_ask:,.2f}")
    print(f"Spread: ${spread:.2f} ({spread_bps:.2f} basis points)")
    print(f"Mid Price: ${mid_price:,.2f}")
    print(f"Liquidity Depth (top 0.1%): ${bid_liquidity:,.0f} (Bid) | ${ask_liquidity:,.0f} (Ask)")
    
    live_data_used = False

print("\n" + "="*50)
print(" WHAT THESE METRICS MEAN FOR THE TRADING DESK:")
print(f"• Spread of {spread_bps:.1f} basis points = ${spread:.2f} cost per trade")
if spread_bps < 2:
    print("   Tight spread - good for market making")
else:
    print("   Wider spread - volatility or low liquidity")

print(f"• Liquidity Depth: ${bid_liquidity:,.0f} waiting to buy within 0.1%")
print(f"• Liquidity Depth: ${ask_liquidity:,.0f} waiting to sell within 0.1%")
if min(bid_liquidity, ask_liquidity) > 500000:
    print("   Deep liquidity - can execute large orders without slippage")
else:
    print("   Shallow liquidity - use limit orders")


if not live_data_used:
    print("\n NOTE: Binance/Bybit APIs were blocked. The logic is identical to live data.")
    print("   In production, these calculations would run on real-time websocket feeds.")

=== Crypto Orderbook Analysis ===
Attempting to fetch live data...

 LIVE DATA from Bybit
Symbol: BTC/USDT
Best Bid: $77,074.50
Best Ask: $77,074.60
Spread: $0.10 (0.01 basis points)
Mid Price: $77,074.55
Liquidity Depth (top 0.1%): $1,057,453 (Bid) | $1,043,115 (Ask)

 WHAT THESE METRICS MEAN FOR THE TRADING DESK:
• Spread of 0.0 basis points = $0.10 cost per trade
   Tight spread - good for market making
• Liquidity Depth: $1,057,453 waiting to buy within 0.1%
• Liquidity Depth: $1,043,115 waiting to sell within 0.1%
   Deep liquidity - can execute large orders without slippage


**Funding Rate Tracker (Binance)**

In [9]:
import requests
import pandas as pd

# Get Binance futures funding rate
url = "https://fapi.binance.com/fapi/v1/premiumIndex?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()

funding_rate = float(data['lastFundingRate'])
next_funding = data['nextFundingTime']
mark_price = float(data['markPrice'])

print(f"BTC/USDT Perpetual")
print(f"Funding Rate: {funding_rate*100:.4f}%")
print(f"Next funding: {next_funding}")
print(f"Mark Price: ${mark_price:,.2f}")

# Calculate annualized funding yield
annualized = funding_rate * 3 * 365  # 3 fundings per day
print(f"Annualized yield (longs paying shorts): {annualized*100:.2f}%")

BTC/USDT Perpetual
Funding Rate: -0.0025%
Next funding: 1777449600000
Mark Price: $77,028.19
Annualized yield (longs paying shorts): -2.77%


**Multi-Exchange Arbitrage Scanner**

In [19]:
import requests
import time
import pandas as pd
from datetime import datetime

# Top 10 tokens by volume
TOKENS = ['BTC', 'ETH', 'SOL', 'XRP', 'DOGE', 'ADA', 'AVAX', 'DOT', 'LINK', 'MATIC']

# Exchange endpoints for ticker prices
def fetch_binance_prices():
    """Get all USDT prices from Binance in 1 request"""
    url = "https://api.binance.com/api/v3/ticker/price"
    response = requests.get(url, timeout=10)
    data = response.json()
    
    prices = {}
    for item in data:
        if item['symbol'].endswith('USDT'):
            token = item['symbol'].replace('USDT', '')
            if token in TOKENS:
                prices[token] = float(item['price'])
    return prices

def fetch_bybit_prices():
    """Get tickers from Bybit"""
    url = "https://api.bybit.com/v5/market/tickers?category=spot"
    response = requests.get(url, timeout=10)
    data = response.json()
    
    prices = {}
    if data['retCode'] == 0:
        for item in data['result']['list']:
            symbol = item['symbol']
            if symbol.endswith('USDT'):
                token = symbol.replace('USDT', '')
                if token in TOKENS:
                    prices[token] = float(item['lastPrice'])
    return prices

def fetch_okx_prices():
    """Get tickers from OKX"""
    url = "https://www.okx.com/api/v5/market/tickers?instType=SPOT"
    response = requests.get(url, timeout=10)
    data = response.json()
    
    prices = {}
    for item in data['data']:
        if item['instId'].endswith('-USDT'):
            token = item['instId'].replace('-USDT', '')
            if token in TOKENS:
                prices[token] = float(item['last'])
    return prices

def fetch_gateio_prices():
    """Get tickers from Gate.io"""
    url = "https://api.gateio.ws/api/v4/spot/tickers"
    response = requests.get(url, timeout=10)
    data = response.json()
    
    prices = {}
    for item in data:
        if item['currency_pair'].endswith('_USDT'):
            token = item['currency_pair'].replace('_USDT', '')
            if token in TOKENS:
                prices[token] = float(item['last'])
    return prices

def fetch_kucoin_prices():
    """Get tickers from KuCoin"""
    url = "https://api.kucoin.com/api/v1/prices"
    response = requests.get(url, timeout=10)
    data = response.json()
    
    prices = {}
    for token, price in data['data'].items():
        if token in TOKENS:
            prices[token] = float(price)
    return prices

print("=" * 80)
print("MULTI-EXCHANGE ARBITRAGE SCANNER")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

# Fetch from all exchanges
print("\nFetching data...")

exchanges = {
    'Binance': fetch_binance_prices,
    'Bybit': fetch_bybit_prices,
    'OKX': fetch_okx_prices,
    'Gate.io': fetch_gateio_prices,
    'KuCoin': fetch_kucoin_prices
}

all_prices = {}
for name, fetch_func in exchanges.items():
    try:
        prices = fetch_func()
        all_prices[name] = prices
        print(f" {name}: {len(prices)} tokens found")
    except Exception as e:
        print(f" {name}: Failed ({str(e)[:50]})")
        all_prices[name] = {}

print("\n" + "=" * 80)
print("ARBITRAGE OPPORTUNITIES (>0.3% spread)")
print("=" * 80)

opportunities = []

for token in TOKENS:
    token_prices = {}
    for exchange, prices in all_prices.items():
        if token in prices:
            token_prices[exchange] = prices[token]
    
    if len(token_prices) >= 2:
        highest_ex = max(token_prices.items(), key=lambda x: x[1])
        lowest_ex = min(token_prices.items(), key=lambda x: x[1])
        spread_pct = ((highest_ex[1] - lowest_ex[1]) / lowest_ex[1]) * 100
        
        if spread_pct > 0.3:
            opportunities.append({
                'Token': token,
                'Buy_Exchange': lowest_ex[0],
                'Buy_Price': lowest_ex[1],
                'Sell_Exchange': highest_ex[0],
                'Sell_Price': highest_ex[1],
                'Spread_%': round(spread_pct, 3),
                'Profit_USD': round(highest_ex[1] - lowest_ex[1], 2)
            })
            print(f" {token}: {spread_pct:.2f}% spread")
            print(f"   Buy: {lowest_ex[0]} @ ${lowest_ex[1]:,.4f}")
            print(f"   Sell: {highest_ex[0]} @ ${highest_ex[1]:,.4f}\n")

if not opportunities:
    print("No opportunities >0.3% found.")
    print("\nCurrent spreads (for reference):")
    for token in TOKENS[:5]:  # Show top 5
        token_prices = {}
        for exchange, prices in all_prices.items():
            if token in prices:
                token_prices[exchange] = prices[token]
        if len(token_prices) >= 2:
            values = list(token_prices.values())
            spread = ((max(values) - min(values)) / min(values)) * 100
            print(f"   {token}: {spread:.3f}% spread across {len(token_prices)} exchanges")

print("\n" + "=" * 80)
print("SUMMARY BY EXCHANGE")
print("=" * 80)

for exchange, prices in all_prices.items():
    if prices:
        print(f"{exchange}: {len(prices)} tokens priced")

MULTI-EXCHANGE ARBITRAGE SCANNER
2026-04-29 08:57:58

Fetching data...
 Binance: 10 tokens found
 Bybit: 9 tokens found
 OKX: 9 tokens found
 Gate.io: 9 tokens found
 KuCoin: 9 tokens found

ARBITRAGE OPPORTUNITIES (>0.3% spread)
No opportunities >0.3% found.

Current spreads (for reference):
   BTC: 0.039% spread across 5 exchanges
   ETH: 0.041% spread across 5 exchanges
   SOL: 0.032% spread across 5 exchanges
   XRP: 0.022% spread across 5 exchanges
   DOGE: 0.087% spread across 5 exchanges

SUMMARY BY EXCHANGE
Binance: 10 tokens priced
Bybit: 9 tokens priced
OKX: 9 tokens priced
Gate.io: 9 tokens priced
KuCoin: 9 tokens priced


**Backtesting Framework (SMA)**

In [15]:
import pandas as pd
import numpy as np

# Simulate price data
np.random.seed(42)
dates = pd.date_range('2024-01-01', '2024-12-31', freq='1min')
returns = np.random.normal(0.00001, 0.0005, len(dates))  # 0.001% mean, 0.05% vol
price = 40000 * np.exp(np.cumsum(returns))

# Convert to pandas Series (this fixes the error)
price_series = pd.Series(price, index=dates)

# Simple moving average crossover strategy
sma_fast = price_series.rolling(10).mean()
sma_slow = price_series.rolling(30).mean()

# Generate signals (1 = buy, -1 = sell, 0 = hold)
positions = []
signal = 0
for i in range(len(price_series)):
    if sma_fast.iloc[i] > sma_slow.iloc[i] and signal <= 0:
        signal = 1
        positions.append(signal)
    elif sma_fast.iloc[i] < sma_slow.iloc[i] and signal >= 0:
        signal = -1
        positions.append(signal)
    else:
        positions.append(0)

positions_series = pd.Series(positions, index=dates)

# Calculate returns
strategy_returns = positions_series.shift(1) * returns
cumulative_strategy = (1 + strategy_returns).cumprod()
cumulative_hold = (1 + returns).cumprod()

print(f"Strategy Return: {(cumulative_strategy.iloc[-1] - 1) * 100:.2f}%")
print(f"Buy & Hold Return: {(cumulative_hold.iloc[-1] - 1) * 100:.2f}%")

# Calculate Sharpe ratio (risk-adjusted return)
strategy_daily = strategy_returns.resample('D').sum()
sharpe = strategy_daily.mean() / strategy_daily.std() * np.sqrt(252) if strategy_daily.std() != 0 else 0
print(f"Sharpe Ratio (annualized): {sharpe:.2f}")

Strategy Return: 0.68%


AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

**SQLite Database Storage**

In [23]:
import sqlite3
import pandas as pd
from datetime import datetime

conn = sqlite3.connect('arbitrage_data.db')
cursor = conn.cursor()

# Drop old table and recreate with 6 columns
cursor.execute('DROP TABLE IF EXISTS spreads')

cursor.execute('''
CREATE TABLE spreads (
    timestamp DATETIME,
    token TEXT,
    spread_pct REAL,
    best_bid_exchange TEXT,
    best_ask_exchange TEXT,
    liquidity_depth_usd REAL
)
''')

# Insert with 6 values
cursor.execute('''
INSERT INTO spreads VALUES (
    datetime('now'),
    'BTC',
    0.032,
    'Binance',
    'OKX',
    2100000
)
''')

conn.commit()

# Verify
df = pd.read_sql_query("SELECT * FROM spreads", conn)
print(df)

conn.close()

             timestamp token  spread_pct best_bid_exchange best_ask_exchange  \
0  2026-04-29 08:00:43   BTC       0.032           Binance               OKX   

   liquidity_depth_usd  
0            2100000.0  
